# 04 · RQ4 news and memes

**Question.** Do news-driven trends last longer, or spread further, than memes? It decides how much the forecasts should lean on news signals (`news_count`, later GDELT), and whether recurring calendar moments deserve their own treatment in the weeks horizon.

**Data.** The stored lists (`trend_items`) through RQ6's matching. Once topic snapshots exist, repeat this with each topic's `news_count`, and later with GDELT's news volume (task 5.8).

**Categories** (`topnews.news`):
- `google`: Google Trends' own trends. Every one comes with a news story, so Google is news by construction;
- `calendar`: recurring calendar moments: weekday hashtags (#WIPWednesday), "national … day", a season's first day;
- `news-linked`: matched to a Google trend at 0.70, so a news story stands behind it;
- `other`: games, tech stories, memes and culture.

Lifespans come from RQ1's spells (Kaplan–Meier), and spread from RQ6's matches. Spread counts lists besides Google, because `news-linked` means matched to Google.

**Needs** `npm ci` at the repo root, as notebook 06 does. Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import pandas as pd

from topnews import db, echo, lifecycle, news, rhythms

pd.set_option("display.max_colwidth", 50)
engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = db.trend_items(engine, days=28)  # YouTube left out
cp, dropped = echo.analyze(items, name="rq4")
categorized = news.categorize(cp)
pd.crosstab(categorized.source_id, categorized.category)

## The labels, for checking

Every calendar and news-linked trend. A wrong one here is a wrong label below.

In [ ]:
for category in ["calendar", "news-linked"]:
    print(category)
    display(categorized[categorized.category == category][["source_id", "title", "news_sim", "best_platform", "best_title"]].round(2))

## Spread

The share of each category's trends also on another list besides Google. Count stories, not trends, before reading much into it: one story can bring several variants (Pinterest's "first day of fall" searches).

In [ ]:
news.spread_by_category(categorized).round(2)

## Lifespans

Time in each list's top 10 by category, strict and with one missed fetch allowed (Bluesky flickers, RQ1). Slow sources are left out, as in RQ1. `inf` means fewer than half the spells have ended yet.

In [ ]:
fast = items[~items.source_id.isin(rhythms.SLOW_SOURCES)]
pd.concat(
    {
        "strict": news.lifespans(news.with_spells(lifecycle.spells(fast), categorized)).set_index(["source_id", "category"]),
        "one missed fetch allowed": news.lifespans(news.with_spells(lifecycle.spells(fast, max_missed=1), categorized)).set_index(["source_id", "category"]),
    },
    axis=1,
).round(2)